In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [5]:
# 1. 宇宙船タイタニックのデータを読み込む
train_data = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')
test_data = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')

In [6]:
print('最初の5行')
display(train_data.head(5))

最初の5行


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


In [7]:
print("データの欠損総和")
print(train_data.isnull().sum())


データの欠損総和
PassengerId       0
HomePlanet       87
CryoSleep        93
Cabin           100
Destination      92
Age              91
VIP              93
RoomService      82
FoodCourt       106
ShoppingMall     98
Spa             101
VRDeck           80
Name             94
dtype: int64


In [8]:
# --- 宇宙船タイタニックの下ごしらえ（データハンドリング） ---

# 1. 数字のデータの穴埋め（年齢や、船内で使った金額など）
# 極端な数値に引っ張られないように、前回と同じく「中央値（median）」で一気に埋めます
num_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_features:
    train_data[col] = train_data[col].fillna(train_data[col].median())
    test_data[col] = test_data[col].fillna(test_data[col].median())

# 2. 文字・カテゴリデータの穴埋め（コールドスリープ中か、VIPか、出身星か）
# これらは数字ではないので、一番データ数の多い「最頻値（mode）」で埋めてあげます
cat_features = ['CryoSleep', 'VIP', 'HomePlanet', 'Destination']
for col in cat_features:
    train_data[col] = train_data[col].fillna(train_data[col].mode()[0])
    test_data[col] = test_data[col].fillna(test_data[col].mode()[0])

# 3. ちゃんと穴が埋まったかもう一度確認！
print("--- 穴埋め後の空欄の数（0になっていれば大成功！） ---")
print(train_data[num_features + cat_features].isnull().sum())

--- 穴埋め後の空欄の数（0になっていれば大成功！） ---
Age             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
CryoSleep       0
VIP             0
HomePlanet      0
Destination     0
dtype: int64


/tmp/ipykernel_57/688372652.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_data[col] = train_data[col].fillna(train_data[col].mode()[0])
/tmp/ipykernel_57/688372652.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_data[col] = test_data[col].fillna(test_data[col].mode()[0])


In [9]:
# --- データの数値化（エンコーディング） ---

# 1. AIに手掛かりとして使わせる項目（特徴量）を選ぶ
features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 
            'CryoSleep', 'VIP', 'HomePlanet', 'Destination']

# 2. pd.get_dummies を使って、文字のデータを自動で 0 と 1 の数字に分解・変換する
X_train = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

# 3. AIに当てさせる「正解ラベル（異次元に飛ばされたか）」を取り出す
# （True/False のままだと扱いにくいので、.astype(int) で 1 と 0 に変換します）
y_train = train_data['Transported'].astype(int)

print("--- 数値化されたデータの見た目（最初の3行） ---")
display(X_train.head(3))

KeyError: 'Transported'

In [ ]:
# 今読み込んでいるデータの「列の名前」をすべて書き出すコード
print(train_data.columns.tolist())

In [10]:
import pandas as pd
import numpy as np
import warnings
# 将来の仕様変更に関する警告を非表示にして画面をスッキリさせる
warnings.simplefilter('ignore', FutureWarning)

print("① 宇宙船タイタニックのデータを真っ新な状態から読み込み中...")
train_data = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')

print("② データの穴埋め（下ごしらえ）を実行中...")
# 【数字データ】は「中央値（median）」で穴埋め
num_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_features:
    train_data[col] = train_data[col].fillna(train_data[col].median())
    test_data[col] = test_data[col].fillna(test_data[col].median())

# 【文字データ】は一番多い「最頻値（mode）」で穴埋め
cat_features = ['CryoSleep', 'VIP', 'HomePlanet', 'Destination']
for col in cat_features:
    train_data[col] = train_data[col].fillna(train_data[col].mode()[0])
    test_data[col] = test_data[col].fillna(test_data[col].mode()[0])

print("③ AIが計算できるようにデータを数値化（エンコーディング）中...")
# AIの手がかりにする項目（特徴量）のリスト
features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 
            'CryoSleep', 'VIP', 'HomePlanet', 'Destination']

# pd.get_dummies を使って文字データを 0 と 1 に自動分解
X_train = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

# AIが当てる「正解（異次元に飛ばされたか：Transported）」を 1 と 0 に変換して取り出す
y_train = train_data['Transported'].astype(int)

print("\n★【大成功】すべての下ごしらえが完了しました！AIに渡すデータの最初の3行です：")
display(X_train.head(3))

① 宇宙船タイタニックのデータを真っ新な状態から読み込み中...


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/spaceship-titanic/train.csv'

In [11]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter('ignore', FutureWarning)

# 1. Load Data
train_data = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')

# 2. Impute Missing Values
num_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in num_features:
    train_data[col] = train_data[col].fillna(train_data[col].median())
    test_data[col] = test_data[col].fillna(test_data[col].median())

cat_features = ['CryoSleep', 'VIP', 'HomePlanet', 'Destination']
for col in cat_features:
    train_data[col] = train_data[col].fillna(train_data[col].mode()[0])
    test_data[col] = test_data[col].fillna(test_data[col].mode()[0])

# 3. One-Hot Encoding & Target Extraction
features = num_features + cat_features

X_train = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])
y_train = train_data['Transported'].astype(int)

# 4. Align Columns (Fill missing columns after get_dummies if any)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

display(X_train.head(3))

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,CryoSleep,VIP,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,39.0,0.0,0.0,0.0,0.0,0.0,False,False,False,True,False,False,False,True
1,24.0,109.0,9.0,25.0,549.0,44.0,False,False,True,False,False,False,False,True
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,True,False,False,False,True


In [12]:
from sklearn.ensemble import RandomForestClassifier

# 1. Initialize Random Forest Model
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)

# 2. Train Model
model.fit(X_train, y_train)

# 3. Predict on Test Data
predictions = model.predict(X_test)

# 4. Shape of predictions
print(f"Predictions calculated: {len(predictions)} rows")

Predictions calculated: 4277 rows


In [13]:
# 1. 予測結果（1 / 0）を True / False に変換
submission_preds = predictions.astype(bool)

# 2. 提出用のデータフレームを作成
output = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Transported': submission_preds
})

# 3. CSVファイルとして出力
output.to_csv('submission.csv', index=False)
print("submission.csv successfully saved!")

submission.csv successfully saved!
